# pricing: out-of-fold E5 description prices
Fixed 10% raw database sample; Isolation Forest auto on training OOF description prices. E5-base reads free-text descriptions only and learns log-price regression. Three-fold predictions create predicted_text_price without exposing each row’s target. Make and model are categorical fields only; there is no make/model transformer. Huber LightGBM combines predicted_text_price with structured fields.

Choose a CUDA runtime. This is a substantial multi-model fine-tuning run. No 10%-sample E5 run has been completed in the assistant environment.


In [ ]:
!git clone https://github.com/botsandbolts/Pricing.git /content/pricing
%cd /content/pricing
!pip -q install -r requirements-training.txt
import torch
assert torch.cuda.is_available(), "Choose a GPU runtime"


In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
source=Path("/content/vehicles.zip")
if not source.exists():
    urlretrieve("https://zenodo.org/records/10457828/files/Used%20Cars%20Dataset.zip?download=1",source)
!python -m scripts.prepare_stack /content/vehicles.zip --fraction .1 --output training-cache/ten-percent.jsonl


In [ ]:
!python -m pricing.train --data training-cache/ten-percent.jsonl --output artifacts/car-price-studio --device cuda --folds 3 --epochs 2
assert Path("artifacts/car-price-studio/manifest.json").exists(), "Training did not finish; inspect the log"


Evaluation uses chronological training OOF prices and train-only models for validation/test. Only after scoring does a fresh OOF pass across all eligible sampled rows and the auto filter train the deployment stack. The assessment and all-row OOF CSVs record the fold for every prediction. Completed fold caches resume only with matching data/configuration. Final all-row models have no independent test set; reported scores belong to assessment.


In [ ]:
import json,shutil
print(json.dumps(json.loads(Path("artifacts/car-price-studio/evaluation.json").read_text()),indent=2))
archive=shutil.make_archive("/content/pricing-e5-oof-all","zip","artifacts")
from google.colab import files
files.download(archive)
